# CharXiv Modeling — Supplementary TabPFN Check (single train/validation split)

Checkpoint 4 and the follow-up bootstrap stability check (notebook 18) both found that no tuned tree
ensemble beats the regularized linear model on the CharXiv failure-prediction task, with cross-validated
ROC-AUC around 0.68 for `GPT-4o` and 0.63 for `Claude-3-5-Sonnet`. That pattern suggests a
feature-to-signal ceiling rather than a model-choice problem. This notebook runs one more check with a
fundamentally different and more flexible model: TabPFN, a pretrained transformer for tabular data that
produces predictions in a single forward pass with no hyperparameter tuning, fit **separately for each
target model**.

Because TabPFN is a transformer and this machine has no GPU, the check uses a single train/validation
split rather than cross-validation or bootstrapping. To keep the comparison fair, the incumbent models
are evaluated on the exact same split at their notebook-17 hyperparameters, since a single-split number
is not comparable to notebook 17's cross-validated means. The validation set is carved out of the 800
training items; the sealed 200-item test set is never read. This is supplementary evidence and does not
change the Checkpoint 4 selection.

This notebook runs in the `charxiv-model` conda environment (it needs TabPFN and torch). On this machine
the duplicate OpenMP runtime requires `KMP_DUPLICATE_LIB_OK=TRUE`.

## 1. Setup

In [1]:
import os
# Guards for the duplicate OpenMP runtime on this Intel Mac: allow the duplicate load and pin the math
# libraries to a single thread (the threadpool/OpenMP conflict segfaults the kernel otherwise). Set
# before importing numpy, torch, or TabPFN.
for _k, _v in {"KMP_DUPLICATE_LIB_OK": "TRUE", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
               "TABPFN_ALLOW_CPU_LARGE_DATASET": "1"}.items():
    os.environ.setdefault(_k, _v)
import sys, warnings, ast, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src.features.preprocessing as P
from src.models import registry
from tabpfn import TabPFNClassifier

RESULTS = ROOT / "results"
TAB = RESULTS / "tabpfn"; TAB.mkdir(parents=True, exist_ok=True)
SEED = 20260618   # same seed lineage as the fixed 800/200 split
TARGETS = P.TARGET_MODELS
REAL = ["GPT-4o", "Claude-3-5-Sonnet"]

con = P.connect(); train_ids, test_ids = P.get_split(con)
df = P.make_design_matrix(con, train_ids)   # one row per item
con.close()
print("import tabpfn OK | items:", len(df))

import tabpfn OK | items: 800


## 2. The single paper-grouped train/validation split

One 80/20 split of the 800 training items, grouped by `paperid` and stratified by the `GPT-4o` failure
label, so no paper (hence no item) straddles the split. The same item split is reused for every target;
the sealed 200-item test set is untouched.

In [2]:
sgk = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr, va = next(sgk.split(df, df[P.ycol("GPT-4o")], df.paperid))
d_tr, d_va = df.iloc[tr].reset_index(drop=True), df.iloc[va].reset_index(drop=True)

assert set(d_tr.item_id).isdisjoint(set(d_va.item_id)), "an item straddles the train/val split!"
assert set(df.item_id).isdisjoint(set(test_ids)), "a sealed test id entered the design matrix!"
print(f"train items {len(d_tr)} | val items {len(d_va)}")
display(d_va[P.TARGET_YCOLS].mean().round(3).rename("val_failure_rate").to_frame())

train items 641 | val items 159


,val_failure_rate
y__GPT-4o,0.535
y__Claude-3-5-Sonnet,0.428
y__GPT-4o-Random,0.899


## 3. Models compared on the identical split

Every model uses the same preprocessing-plus-estimator pipeline as the rest of the project, so the
feature set is identical across models. TabPFN is used as-is with no tuning. For the two real targets
the incumbents are rebuilt at their notebook-17 winning hyperparameters, read from
`results/model_comparison.csv`. `GPT-4o-Random` has no tuned incumbents (it is never tuned), so it is
compared against the default Random Forest and Logistic Regression baselines.

In [3]:
def tuned_factory(family, modal_params):
    params = {k.replace("clf__", ""): v for k, v in ast.literal_eval(modal_params).items()}
    def factory():
        est = registry.FACTORIES[family]()
        est.set_params(**params)
        return est
    return factory

mc = pd.read_csv(RESULTS / "model_comparison.csv")
modal = {(r.target, r.family): r.modal_best_params
         for _, r in mc.query("config == 'tuned'").iterrows()}

def tabpfn_factory():
    return TabPFNClassifier(random_state=0, ignore_pretraining_limits=True)

def build_models(target):
    """Return {name: (factory, label)} for a target: TabPFN, the incumbents, and default logistic."""
    models = {"tabpfn": (tabpfn_factory, "TabPFN (pretrained, untuned)")}
    if target in REAL:
        for fam in ["logistic_regularized", "random_forest", "hist_gb", "xgboost"]:
            models[fam] = (tuned_factory(fam, modal[(target, fam)]), f"{fam} (nb17 tuned)")
    else:
        models["random_forest"] = (registry.FACTORIES["random_forest"], "random_forest (default)")
    models["logistic"] = (registry.FACTORIES["logistic"], "logistic (default baseline)")
    return models

print("real-target incumbents:", [f for f in ["logistic_regularized","random_forest","hist_gb","xgboost"]])

real-target incumbents: ['logistic_regularized', 'random_forest', 'hist_gb', 'xgboost']


## 4. Fit on train, score on the held-out validation rows, per target

For each target, each model is fit on the training items and scored on the validation items with the
project's three key performance indicators: ROC-AUC (primary), and balanced accuracy and F1 at the 0.5
threshold. TabPFN's fit and predict is the slow step on CPU.

In [4]:
def kpis(y_true, prob, threshold=0.5):
    yhat = (prob >= threshold).astype(int)
    return {"roc_auc": roc_auc_score(y_true, prob),
            "bal_acc": balanced_accuracy_score(y_true, yhat),
            "f1": f1_score(y_true, yhat, zero_division=0)}

rows = []
for target in TARGETS:
    y_tr, y_va = d_tr[P.ycol(target)].to_numpy(), d_va[P.ycol(target)].to_numpy()
    for name, (factory, label) in build_models(target).items():
        t0 = time.time()
        pipe = Pipeline([("pre", P.build_preprocessor()), ("clf", factory())]).fit(d_tr, y_tr)
        prob = pipe.predict_proba(d_va)[:, 1]
        m = kpis(y_va, prob)
        rows.append({"target": target, "model": name, "label": label,
                     "fit_predict_s": round(time.time() - t0, 1), **{k: round(v, 4) for k, v in m.items()}})
    print(f"done: {target}")

table = pd.DataFrame(rows)[["target", "model", "label", "roc_auc", "bal_acc", "f1", "fit_predict_s"]]
table.to_csv(TAB / "tabpfn_single_split.csv", index=False)
print("wrote", TAB / "tabpfn_single_split.csv")
for target in TARGETS:
    print(f"=== {target} ===")
    display(table[table.target == target].sort_values("roc_auc", ascending=False)
            .set_index("model")[["roc_auc", "bal_acc", "f1", "fit_predict_s"]])

done: GPT-4o


done: Claude-3-5-Sonnet


done: GPT-4o-Random
wrote /Users/everett/Documents/GitHub/summer26-ai-faithfulness-in-scientific-reasoning-private/results/tabpfn/tabpfn_single_split.csv
=== GPT-4o ===


,roc_auc,bal_acc,f1,fit_predict_s
model,,,,
hist_gb,0.6341,0.5788,0.5939,0.1
xgboost,0.6335,0.6032,0.6228,0.1
tabpfn,0.6238,0.5703,0.6000,22.5
random_forest,0.6130,0.5441,0.5814,0.3
logistic,0.6103,0.5770,0.6036,0.0
logistic_regularized,0.6093,0.5703,0.6000,0.0


=== Claude-3-5-Sonnet ===


,roc_auc,bal_acc,f1,fit_predict_s
model,,,,
tabpfn,0.6136,0.5719,0.3711,21.5
logistic_regularized,0.6018,0.5976,0.4124,0.0
random_forest,0.6004,0.5003,0.2500,0.3
logistic,0.5984,0.5940,0.4200,0.0
xgboost,0.5847,0.5206,0.3269,0.0
hist_gb,0.5669,0.5520,0.4274,0.1


=== GPT-4o-Random ===


,roc_auc,bal_acc,f1,fit_predict_s
model,,,,
tabpfn,0.5835,0.5,0.947,21.5
random_forest,0.5721,0.5,0.947,0.4
logistic,0.5035,0.5,0.947,0.0


## 5. TabPFN versus the best incumbent, per target

For each target the table shows TabPFN's validation ROC-AUC next to the best incumbent on the identical
split, and the gap. With a single validation fold of about 155 items the key performance indicators
carry real sampling variance, so a gap smaller than about 0.02 ROC-AUC should be read as a tie.

In [5]:
summary = []
for target in TARGETS:
    sub = table[table.target == target]
    tab = float(sub[sub.model == "tabpfn"]["roc_auc"].iloc[0])
    inc = sub[sub.model != "tabpfn"]
    best = inc.sort_values("roc_auc", ascending=False).iloc[0]
    summary.append({"target": target, "tabpfn_roc_auc": tab,
                    "best_incumbent": best.model, "incumbent_roc_auc": float(best.roc_auc),
                    "tabpfn_minus_incumbent": round(tab - float(best.roc_auc), 4)})
display(pd.DataFrame(summary).set_index("target"))

,tabpfn_roc_auc,best_incumbent,incumbent_roc_auc,tabpfn_minus_incumbent
target,,,,
GPT-4o,0.6238,hist_gb,0.6341,-0.0103
Claude-3-5-Sonnet,0.6136,logistic_regularized,0.6018,0.0118
GPT-4o-Random,0.5835,random_forest,0.5721,0.0114


## 6. Findings

Read the per-target comparison in sections 4 and 5. The question is whether TabPFN's validation ROC-AUC
clears the incumbents on this shared split. With a single validation fold of roughly 155 items the key
performance indicators carry real sampling variance, so a gap smaller than about 0.02 ROC-AUC should be
read as a tie rather than a genuine improvement. Scoring the incumbents on the exact same split is what
makes the comparison fair; the incumbent numbers here differ from notebook 17's cross-validated means
only because this is one fold rather than a fifteen-fold average.

If TabPFN lands within that noise band of the incumbents for the real targets, the supplementary check
reinforces the Checkpoint 4 conclusion that the binding constraint is the information in the features,
not the flexibility of the model. For `GPT-4o-Random` every model stays weak, as expected for the
random-answer control. Either way this single-split test is supplementary and does not by itself
override the cross-validated and bootstrapped results.

In [6]:
# Guardrail: the sealed test set never entered training, validation, or the design matrix.
assert set(d_tr.item_id).isdisjoint(set(test_ids))
assert set(d_va.item_id).isdisjoint(set(test_ids))
print("OK: the 200 sealed test ids never entered the training or validation matrices.")

OK: the 200 sealed test ids never entered the training or validation matrices.
